# Oregon — ORS Volume 18 (insurance chapters) → `data/oregon/ins_codes/*.md`

On **Justia**, **Oregon Revised Statutes** are organized by **volume**, not a single “insurance title” URL. **Volume 18 — Financial Institutions, Insurance** includes banking chapters (**705–727**) and **insurance-related chapters from 731 onward** (administration through **752**).

This notebook downloads **Volume 18 chapters whose numeric prefix is ≥ 731** (~**24** chapters, ~**1,565** sections), e.g. **[`/codes/oregon/volume-18/chapter-743/`](https://law.justia.com/codes/oregon/volume-18/chapter-743/)** with sections like **`…/section-743-004/`**.

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** GET the **[volume-18 index](https://law.justia.com/codes/oregon/volume-18/)** (**`div.primary-content`**), collect **`chapter-*`** links under **`/codes/oregon/volume-18/`** with chapter number **≥ `CHAPTER_NUM_MIN`**, then GET each chapter page and collect **`section-*`** URLs under the same volume path.

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`OR_sec_<slug>.md`**. Display cites **O.R.S. §** … (e.g. slug **`743-004`** → **`O.R.S. § 743.004`**, **`743a-001`** → **`O.R.S. § 743A.001`**).

Config: **CHAPTER_NUM_MIN** (default **731**), **MAX_SECTIONS** (**0** = all). **REUSE_DISCOVERED_URLS** skips discovery when **`_or_vol18_ins_section_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/oregon/volume-18"
VOLUME_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "oregon" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

CHAPTER_NUM_MIN = 731

MAX_SECTIONS = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_or_vol18_ins_section_urls.txt"
REUSE_DISCOVERED_URLS = True


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def chapter_leading_int(chapter_path: str) -> int | None:
    m = re.search(r"/chapter-([^/]+)/", chapter_path + "/", re.I)
    if not m:
        return None
    m2 = re.match(r"^(\d+)", m.group(1))
    return int(m2.group(1)) if m2 else None


def discover_chapter_urls() -> list[str]:
    html = curl_get(VOLUME_INDEX)
    soup = BeautifulSoup(html, "html.parser")
    pc = soup.select_one("div.primary-content")
    if not pc:
        pc = soup
    chapters: set[str] = set()
    for a in pc.find_all("a", href=True):
        absu = urljoin(VOLUME_INDEX, a["href"])
        pk = path_key(absu).lower()
        if not pk.startswith(PATH_PREFIX.lower() + "/chapter-"):
            continue
        n = chapter_leading_int(pk)
        if n is None or n < CHAPTER_NUM_MIN:
            continue
        chapters.add(absu if absu.endswith("/") else absu + "/")
    return sorted(chapters, key=lambda u: (chapter_leading_int(path_key(u)) or 0, path_key(u).lower()))


def section_urls_on_chapter_page(chapter_url: str) -> list[str]:
    html = curl_get(chapter_url)
    soup = BeautifulSoup(html, "html.parser")
    pc = soup.select_one("div.primary-content")
    if not pc:
        pc = soup
    ch_pk = path_key(chapter_url).lower()
    found: set[str] = set()
    for a in pc.find_all("a", href=True):
        absu = urljoin(chapter_url, a["href"])
        pk = path_key(absu).lower()
        if "/section-" not in pk:
            continue
        if not pk.startswith(PATH_PREFIX.lower()):
            continue
        if not pk.startswith(ch_pk + "/"):
            continue
        found.add(absu if absu.endswith("/") else absu + "/")
    return sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))


def discover_section_urls() -> list[str]:
    """Volume 18 index → chapter pages (ch ≥ CHAPTER_NUM_MIN) → section URLs."""
    chapters = discover_chapter_urls()
    print(f"Chapters to scan: {len(chapters)} (chapter number ≥ {CHAPTER_NUM_MIN})")
    all_secs: list[str] = []
    seen: set[str] = set()
    for i, ch in enumerate(chapters, 1):
        for su in section_urls_on_chapter_page(ch):
            if su not in seen:
                seen.add(su)
                all_secs.append(su)
        if i % 6 == 0:
            print(f"… discovery {i}/{len(chapters)} chapters, {len(all_secs)} sections so far")
    return sorted(all_secs, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    return path.rsplit("/section-", 1)[1]


def label_sort_key(label: str) -> tuple:
    m = re.match(r"^(\d+)([a-z]*)-(.+)$", label, re.I)
    if not m:
        out: list[tuple[int, int | str]] = []
        for part in label.split("-"):
            if part.isdigit():
                out.append((0, int(part)))
            else:
                out.append((1, part.lower()))
        return tuple(out)
    n = int(m.group(1))
    suf = m.group(2).lower()
    sec = m.group(3)
    sec_parts: list[tuple[int, int | str]] = []
    for p in sec.split("-"):
        if p.isdigit():
            sec_parts.append((0, int(p)))
        else:
            sec_parts.append((1, p.lower()))
    return ((0, n), (1, suf), *sec_parts)


def label_to_display_citation(label: str) -> str:
    """743-004 → O.R.S. § 743.004 ; 743a-001 → O.R.S. § 743A.001"""
    m = re.match(r"^(\d+)([a-z]*)-(.+)$", label, re.I)
    if not m:
        return label
    chap = m.group(1) + m.group(2).upper()
    sec = m.group(3)
    return f"O.R.S. § {chap}.{sec}"


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"OR_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("Oregon Rev" in s or "Oregon Revised" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("Oregon Rev. Stat."):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_oregon_ins_volume18() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs (Volume 18, chapters ≥ {CHAPTER_NUM_MIN})")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Oregon Revised Statutes {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Oregon Revised Statutes — Volume 18 (insurance chapters, chapter ≥ {CHAPTER_NUM_MIN})**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [Oregon Revised Statutes (Oregon Legislature)](https://www.oregonlegislature.gov/bills_laws/pages/oregon_revised_statutes.aspx)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_oregon_ins_volume18()


Chapters to scan: 24 (chapter number ≥ 731)
… discovery 6/24 chapters, 608 sections so far
… discovery 12/24 chapters, 837 sections so far
… discovery 18/24 chapters, 1460 sections so far
… discovery 24/24 chapters, 1565 sections so far
Discovered 1565 section URLs (Volume 18, chapters ≥ 731)
… 200/1565 (wrote=200 skipped=0 failed=0)
… 400/1565 (wrote=400 skipped=0 failed=0)
… 600/1565 (wrote=600 skipped=0 failed=0)
… 800/1565 (wrote=800 skipped=0 failed=0)
… 1000/1565 (wrote=1000 skipped=0 failed=0)
… 1200/1565 (wrote=1200 skipped=0 failed=0)
… 1400/1565 (wrote=1400 skipped=0 failed=0)
Done. wrote=1565 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/oregon/ins_codes


{'wrote': 1565, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
